# Phase 1 — Data Inspection
First-pass look at the 2025 Stack Overflow Developer Survey.

**Key 2025 differences from prior years:**
- Files are `.txt` (comma-separated, not `.csv`)
- Multi-select columns use **semicolons** as delimiter (not pipes)
- `YearsCodePro` renamed to **`WorkExp`**
- `FrameworkHaveWorkedWith` renamed to **`WebframeHaveWorkedWith`**

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)

## Load Data

In [2]:
df = pd.read_csv('../data/results.txt', low_memory=False)
schema = pd.read_csv('../data/schema.txt')

print(f'Survey shape : {df.shape}')       # (52845, n_cols)
print(f'Schema shape : {schema.shape}')

Survey shape : (49191, 172)
Schema shape : (139, 6)


## Schema — Human-Readable Column Descriptions

In [3]:
schema.head(10)

,qid,qname,question,type,sub,sq_id
0,QID18,TechEndorse_1,What attracts you to a technology or causes you to endorse it (most to least...,RO,AI integration or AI Agent capabilities,1.0
1,QID18,TechEndorse_2,What attracts you to a technology or causes you to endorse it (most to least...,RO,Easy-to-use API,2.0
2,QID18,TechEndorse_3,What attracts you to a technology or causes you to endorse it (most to least...,RO,Robust and complete API,3.0
3,QID18,TechEndorse_4,What attracts you to a technology or causes you to endorse it (most to least...,RO,Customizable and manageable codebase,4.0
4,QID18,TechEndorse_5,What attracts you to a technology or causes you to endorse it (most to least...,RO,Reputation for quality,5.0
5,QID18,TechEndorse_6,What attracts you to a technology or causes you to endorse it (most to least...,RO,Connected to an open-source project,6.0
6,QID18,TechEndorse_7,What attracts you to a technology or causes you to endorse it (most to least...,RO,Good brand and public image,7.0
7,QID18,TechEndorse_8,What attracts you to a technology or causes you to endorse it (most to least...,RO,Reliability and low latency,8.0
8,QID18,TechEndorse_9,What attracts you to a technology or causes you to endorse it (most to least...,RO,Costs are manageable,9.0
9,QID18,TechEndorse_13,What attracts you to a technology or causes you to endorse it (most to least...,RO,Other (please specify):,10.0


In [4]:
# Quick lookup: what does a column name mean?
def describe_col(col_name):
    match = schema[schema['qname'] == col_name]
    if match.empty:
        return f'{col_name} — not found in schema'
    return match[['qname', 'question', 'type']].drop_duplicates().to_string(index=False)

print(describe_col('WorkExp'))
print()
print(describe_col('ConvertedCompYearly'))

  qname                                                                                                                                                                        question type
WorkExp How many years of professional work experience do you have? Please round to the nearest whole number, excluding any decimal points.  If your answer is '0', please leave blank.   TE

ConvertedCompYearly — not found in schema


## First Glance

In [5]:
df.head(3)

,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,AILearnHow,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,TechEndorse_1,TechEndorse_2,TechEndorse_3,TechEndorse_4,TechEndorse_5,TechEndorse_6,TechEndorse_7,TechEndorse_8,TechEndorse_9,TechEndorse_13,TechEndorse_13_TEXT,TechOppose_1,...,AIToolPlan to mostly use AI,AIToolCurrently mostly AI,AIFrustration,AIExplain,AIAgents,AIAgentChange,AIAgent_Uses,AgentUsesGeneral,AIAgentImpactSomewhat agree,AIAgentImpactNeutral,AIAgentImpactSomewhat disagree,AIAgentImpactStrongly agree,AIAgentImpactStrongly disagree,AIAgentChallengesNeutral,AIAgentChallengesSomewhat disagree,AIAgentChallengesStrongly agree,AIAgentChallengesSomewhat agree,AIAgentChallengesStrongly disagree,AIAgentKnowledge,AIAgentKnowWrite,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning new coding techniques or program...",Online Courses or Certification (includes all media types);Other online reso...,"Yes, I learned how to use AI-enabled tools for my personal curiosity and/or ...",AI CodeGen tools or AI-enabled apps,14.0,"Developer, mobile",20 to 99 employees,People manager,Remote,"Yes, I influenced the purchase of a substantial addition to the tech stack",Work,10.0,7.0,9.0,6.0,3.0,11.0,12.0,1.0,8.0,14.0,NaN,15.0,...,NaN,NaN,"AI solutions that are almost right, but not quite",No,"Yes, I use AI agents at work monthly or infrequently",Not at all or minimally,Software engineering,NaN,AI agents have increased my productivity.;AI agents have reduced the time sp...,AI agents have helped me automate repetitive tasks.;AI agents have improved ...,NaN,NaN,NaN,I am concerned about the accuracy of the information provided by AI agents.;...,Integrating AI agents with my existing tools and workflows can be difficult....,The cost of using certain AI agent platforms is a barrier.;I have concerns a...,NaN,NaN,NaN,NaN,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning new coding techniques or program...",Online Courses or Certification (includes all media types);Other online reso...,"Yes, I learned how to use AI-enabled tools for my personal curiosity and/or ...",AI CodeGen tools or AI-enabled apps,10.0,"Developer, back-end",500 to 999 employees,Individual contributor,"Hybrid (some in-person, leans heavy to flexibility)",No,Personal Project,13.0,1.0,2.0,9.0,4.0,3.0,12.0,5.0,7.0,14.0,NaN,14.0,...,NaN,NaN,"AI solutions that are almost right, but not quite;Debugging AI-generated cod...",No,"No, and I don't plan to",Not at all or minimally,NaN,NaN,NaN,NaN,NaN,NaN,NaN,It takes significant time and effort to learn how to use AI agents effective...,NaN,I am concerned about the accuracy of the information provided by AI agents.;...,Integrating AI agents with my existing tools and workflows can be difficult....,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to fully understand something;Wh...,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-employed",None of the above,10.0,"Yes, I am not new to coding but am learning new coding techniques or program...",Online Courses or Certification (includes all media types);Videos (not assoc...,"Yes, I learned how to use AI-enabled tools for my personal curiosity and/or ...",AI CodeGen tools or AI-enabled apps;Technical documentation (is generated

In [6]:
# df.info() output is too large to capture; use targeted summary
print('Column dtypes:')
print(df.dtypes.value_counts())
print()
print('Non-null counts for key columns:')
print(df[existing].notna().sum().sort_values())

In [7]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ResponseId,49191.0,NaN,NaN,NaN,24596.0,14200.362883,1.0,12298.5,24596.0,36893.5,49191.0
MainBranch,49191,6,I am a developer by profession,37467,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,49191,7,25-34 years old,16519,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EdLevel,48149,8,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",20278,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Employment,48339,6,Employed,33750,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
AIAgentExtWrite,859,421,Cursor,106,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AIHuman,29194,574,When I don’t trust AI’s answers;When I want to fully understand something;Wh...,3050,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AIOpen,22540,20138,Problem solving,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ConvertedCompYearly,23947.0,NaN,NaN,NaN,101761.539901,461756.901211,1.0,38171.0,75320.0,120596.0,50000000.0


## Target Variable — ConvertedCompYearly

In [8]:
target = 'ConvertedCompYearly'

print(f'Total rows       : {len(df):,}')
print(f'Null salary      : {df[target].isna().sum():,}')
print(f'Non-null salary  : {df[target].notna().sum():,}')
print()
print(df[target].describe())

Total rows       : 49,191
Null salary      : 25,244
Non-null salary  : 23,947

count    2.394700e+04
mean     1.017615e+05
std      4.617569e+05
min      1.000000e+00
25%      3.817100e+04
50%      7.532000e+04
75%      1.205960e+05
max      5.000000e+07
Name: ConvertedCompYearly, dtype: float64


## Key Feature Columns (2025 names)

In [9]:
key_cols = [
    'ConvertedCompYearly',   # target — USD salary
    'WorkExp',               # years of professional coding (was YearsCodePro)
    'YearsCode',             # total years coding
    'Country',
    'DevType',
    'EdLevel',
    'Employment',
    'LanguageHaveWorkedWith',
    'DatabaseHaveWorkedWith',
    'WebframeHaveWorkedWith',  # was FrameworkHaveWorkedWith
]

existing = [c for c in key_cols if c in df.columns]
missing  = [c for c in key_cols if c not in df.columns]

print('Found  :', existing)
print('Missing:', missing)

Found  : ['ConvertedCompYearly', 'WorkExp', 'YearsCode', 'Country', 'DevType', 'EdLevel', 'Employment', 'LanguageHaveWorkedWith', 'DatabaseHaveWorkedWith', 'WebframeHaveWorkedWith']
Missing: []


In [10]:
df[existing].head(10)

,ConvertedCompYearly,WorkExp,YearsCode,Country,DevType,EdLevel,Employment,LanguageHaveWorkedWith,DatabaseHaveWorkedWith,WebframeHaveWorkedWith
0,61256.0,8.0,14.0,Ukraine,"Developer, mobile","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,Bash/Shell (all shells);Dart;SQL,Cloud Firestore;PostgreSQL,NaN
1,104413.0,2.0,10.0,Netherlands,"Developer, back-end","Associate degree (A.A., A.S., etc.)",Employed,Java,Dynamodb;MongoDB,Spring Boot
2,53061.0,10.0,12.0,Ukraine,"Developer, front-end","Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-employed",Dart;HTML/CSS;JavaScript;TypeScript,MongoDB;MySQL;PostgreSQL,Next.js;Node.js;React
3,36197.0,4.0,5.0,Ukraine,"Developer, back-end","Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,Java;Kotlin;SQL,NaN,Spring Boot
4,60000.0,21.0,22.0,Ukraine,Engineering manager,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-employed",C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;PowerShell;Python;SQL;TypeScrip...,Elasticsearch;Microsoft SQL Server;MySQL;Oracle;PostgreSQL;Redis;SQLite,Angular;ASP.NET;ASP.NET Core;Flask;jQuery
5,120000.0,15.0,20.0,Ukraine,"Developer, back-end","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-employed",Java;Scala,NaN,Spring Boot
6,6240.0,9.0,13.0,Ukraine,"Developer, full-stack",Some college/university study without earning a degree,"Independent contractor, freelancer, or self-employed",JavaScript;TypeScript,NaN,Next.js;Node.js;React
7,72000.0,22.0,30.0,Ukraine,"Architect, software or solutions","Professional degree (JD, MD, Ph.D, Ed.D, etc.)",Employed,Bash/Shell (all shells);HTML/CSS;JavaScript;Python;SQL;TypeScript,MongoDB;PostgreSQL;SQLite,Angular;Django
8,70000.0,9.0,15.0,Ukraine,Data engineer,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,Java;Python;Scala,Cassandra;Databricks SQL;DuckDB;Dynamodb;MariaDB;Microsoft SQL Server;MySQL;...,FastAPI;Spring Boot
9,5400.0,7.0,10.0,Ukraine,"Developer, mobile","Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,NaN,NaN,NaN


## Missing Value Summary

In [11]:
missing_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].head(30).to_string())

AIAgentObsWrite                   99.463316
SOTagsWant Entry                  99.125856
SOTagsHaveEntry                   99.068935
AIModelsWantEntry                 99.034376
AIAgentOrchWrite                  99.028278
JobSatPoints_15_TEXT              98.650160
AIAgentKnowWrite                  98.442805
AIModelsHaveEntry                 98.422476
SO_Actions_15_TEXT                98.326930
AIAgentExtWrite                   98.253746
CommPlatformWantEntr              97.593056
CommPlatformHaveEntr              96.999451
DatabaseWantEntry                 96.883576
OfficeStackWantEntry              96.712813
TechOppose_15_TEXT                96.651827
TechEndorse_13_TEXT               95.915920
DevEnvWantEntry                   95.706532
DatabaseHaveEntry                 95.629282
OfficeStackHaveEntry              94.736842
WebframeWantEntry                 94.653494
AIAgentObserveSecure              94.527454
DevEnvHaveEntry                   94.385152
PlatformWantEntry               

In [12]:
# Missing % for our key columns only
key_missing = (df[existing].isna().sum() / len(df) * 100).sort_values(ascending=False)
print('Missing % for key columns:')
print(key_missing.to_string())

Missing % for key columns:
WebframeHaveWorkedWith    53.259743
ConvertedCompYearly       51.318331
DatabaseHaveWorkedWith    48.059604
LanguageHaveWorkedWith    35.616271
Country                   27.960399
WorkExp                   12.803155
YearsCode                 12.500254
DevType                   11.203269
EdLevel                    2.118274
Employment                 1.732024


## Semicolon-Delimited Columns — Sample Values

In [13]:
multi_cols = ['LanguageHaveWorkedWith', 'DatabaseHaveWorkedWith', 'WebframeHaveWorkedWith']

for col in multi_cols:
    if col in df.columns:
        print(f'--- {col} ---')
        print(df[col].dropna().head(5).to_string())
        # Unique individual values
        unique_vals = df[col].dropna().str.split(';').explode().str.strip().value_counts()
        print(f'  → {len(unique_vals)} unique values, top 5: {list(unique_vals.index[:5])}')
        print()

--- LanguageHaveWorkedWith ---
0                                                   Bash/Shell (all shells);Dart;SQL
1                                                                               Java
2                                                Dart;HTML/CSS;JavaScript;TypeScript
3                                                                    Java;Kotlin;SQL
4    C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;PowerShell;Python;SQL;TypeScrip...
  → 42 unique values, top 5: ['JavaScript', 'HTML/CSS', 'SQL', 'Python', 'Bash/Shell (all shells)']

--- DatabaseHaveWorkedWith ---
0                                                 Cloud Firestore;PostgreSQL
1                                                           Dynamodb;MongoDB
2                                                   MongoDB;MySQL;PostgreSQL
4    Elasticsearch;Microsoft SQL Server;MySQL;Oracle;PostgreSQL;Redis;SQLite
7                                                  MongoDB;PostgreSQL;SQLite
  → 30 unique values, top 5

## Unique Values — Categorical Columns

In [14]:
for col in ['Country', 'EdLevel', 'Employment', 'DevType']:
    if col in df.columns:
        print(f'{col}  ({df[col].nunique()} unique):')
        print(df[col].value_counts().head(10).to_string())
        print()

Country  (177 unique):
Country
United States of America                                7233
Germany                                                 3025
India                                                   2547
United Kingdom of Great Britain and Northern Ireland    2042
France                                                  1409
Canada                                                  1305
Ukraine                                                  964
Poland                                                   888
Netherlands                                              867
Italy                                                    835

EdLevel  (8 unique):
EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          20278
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       12589
Some college/university study without earning a degree                                 6182
Secondary school (e.g. American high school, German R

## WorkExp Distribution

In [15]:
print(df['WorkExp'].value_counts().sort_index().to_string())

WorkExp
1.0      2453
2.0      2408
3.0      2613
4.0      2166
5.0      2563
6.0      1810
7.0      1775
8.0      1963
9.0      1221
10.0     2885
11.0     1087
12.0     1587
13.0     1176
14.0      907
15.0     2226
16.0      741
17.0      757
18.0      902
19.0      549
20.0     2094
21.0      457
22.0      518
23.0      378
24.0      385
25.0     1649
26.0      411
27.0      429
28.0      453
29.0      206
30.0     1201
31.0      161
32.0      181
33.0      170
34.0      134
35.0      564
36.0      128
37.0      128
38.0      140
39.0       82
40.0      459
41.0       53
42.0      115
43.0       66
44.0       29
45.0      181
46.0       38
47.0       27
48.0       18
49.0       10
50.0       92
51.0       12
52.0       10
53.0        7
54.0       13
55.0       19
56.0        6
57.0        4
58.0        5
59.0        3
60.0       14
61.0        2
63.0        1
65.0        3
66.0        1
68.0        2
69.0        2
70.0        1
75.0        1
82.0        1
88.0        1
99.0        

## Summary Notes

- **Total respondents:** 49,191
- **Respondents with salary data:** 23,947 (48.7% of total)
- **Salary range (raw):** $1 to $50,000,000 — heavy right skew; needs outlier filtering and log-transform
- **Missing % for key features:** ConvertedCompYearly 51%, WebframeHaveWorkedWith 53%, DatabaseHaveWorkedWith 48%, LanguageHaveWorkedWith 36%, Country 28%
- **ConvertedCompYearly not in schema:** expected — Stack Overflow's derived USD-normalised column, added post-survey
- **WorkExp data quality issue:** 34 rows have WorkExp=100 and 5 rows have WorkExp=99 — clearly erroneous; cap at 50 years in Phase 3 cleaning
- **Delimiter confirmed:** semicolon (`;`) for all multi-select tech columns
- **All 10 key feature columns present:** no missing or unexpected renames